In [1]:
import os
import pandas as pd

print("=== ALL FILES ===")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        path = os.path.join(dirname, filename)
        size = os.path.getsize(path) / 1024**2
        print(f"{size:8.1f} MB  {path}")

print("\n=== STRUCTURE OF EACH CSV ===")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith('.csv'):
            path = os.path.join(dirname, filename)
            try:
                df = pd.read_csv(path, nrows=5)
                print(f"\n📄 {path}")
                print(f"   Columns ({len(df.columns)}): {list(df.columns[:15])}{' ...' if len(df.columns) > 15 else ''}")
                # guess the label column and show its values
                for cand in ['label', 'Label', 'class', 'Class', 'Category',
                             'category', 'Attack', 'attack', 'type', 'malware']:
                    if cand in df.columns:
                        full_labels = pd.read_csv(path, usecols=[cand])
                        print(f"   Label '{cand}': {full_labels[cand].value_counts().head(10).to_dict()}")
                        break
            except Exception as e:
                print(f"\n📄 {path} → could not read: {e}")

=== ALL FILES ===
     2.4 MB  /kaggle/input/datasets/mrmorj/hate-speech-and-offensive-language-dataset/labeled_data.csv
   331.8 MB  /kaggle/input/datasets/himadri07/ciciot2023/CICIOT23/validation/validation.csv
   331.8 MB  /kaggle/input/datasets/himadri07/ciciot2023/CICIOT23/test/test.csv
  1548.2 MB  /kaggle/input/datasets/himadri07/ciciot2023/CICIOT23/train/train.csv
    18.1 MB  /kaggle/input/datasets/luccagodoy/obfuscated-malware-memory-2022-cic/Obfuscated-MalMem2022.csv
    51.7 MB  /kaggle/input/datasets/ndarvind/phiusiil-phishing-url-dataset/PhiUSIIL_Phishing_URL_Dataset.csv

=== STRUCTURE OF EACH CSV ===

📄 /kaggle/input/datasets/mrmorj/hate-speech-and-offensive-language-dataset/labeled_data.csv
   Columns (7): ['Unnamed: 0', 'count', 'hate_speech', 'offensive_language', 'neither', 'class', 'tweet']
   Label 'class': {1: 19190, 2: 4163, 0: 1430}

📄 /kaggle/input/datasets/himadri07/ciciot2023/CICIOT23/validation/validation.csv
   Columns (47): ['flow_duration', 'Header_Length

In [4]:
import pandas as pd
DYNAHATE_URL = ('https://raw.githubusercontent.com/bvidgen/'
                'Dynamically-Generated-Hate-Speech-Dataset/main/'
                'Dynamically%20Generated%20Hate%20Dataset%20v0.2.3.csv')
df_dyna = pd.read_csv(DYNAHATE_URL)
print("✅ DynaHate downloaded!")
print("Shape:", df_dyna.shape)
print(df_dyna['label'].value_counts())

✅ DynaHate downloaded!
Shape: (41144, 13)
label
hate       22175
nothate    18969
Name: count, dtype: int64


In [5]:
import os
import re
import numpy as np
import pandas as pd
import warnings
import gc
import pickle
import time
import json
import torch
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score

# ---- v3 dataset paths (from inspection) ----
NET_TRAIN_PATH = '/kaggle/input/datasets/himadri07/ciciot2023/CICIOT23/train/train.csv'
NET_VAL_PATH   = '/kaggle/input/datasets/himadri07/ciciot2023/CICIOT23/validation/validation.csv'
NET_TEST_PATH  = '/kaggle/input/datasets/himadri07/ciciot2023/CICIOT23/test/test.csv'
PHISHING_PATH  = '/kaggle/input/datasets/ndarvind/phiusiil-phishing-url-dataset/PhiUSIIL_Phishing_URL_Dataset.csv'
MALWARE_PATH   = '/kaggle/input/datasets/luccagodoy/obfuscated-malware-memory-2022-cic/Obfuscated-MalMem2022.csv'
SOCIAL_PATH    = '/kaggle/input/datasets/mrmorj/hate-speech-and-offensive-language-dataset/labeled_data.csv'
DYNAHATE_URL   = ('https://raw.githubusercontent.com/bvidgen/'
                  'Dynamically-Generated-Hate-Speech-Dataset/main/'
                  'Dynamically%20Generated%20Hate%20Dataset%20v0.2.3.csv')

print("✅ CyberWatch AI V3 — paths set!")
print("  Module 1 → CICIoT2023 (XGBoost + LSTM)")
print("  Module 2 → PhiUSIIL 2024 (RF + XGB Ensemble)")
print("  Module 3 → CIC-MalMem-2022 (CNN, 4 classes!)")
print("  Module 4 → Davidson + DynaHate (BERT)")
print("  Anomaly  → CICIoT2023 benign (IF + Autoencoder)")

✅ CyberWatch AI V3 — paths set!
  Module 1 → CICIoT2023 (XGBoost + LSTM)
  Module 2 → PhiUSIIL 2024 (RF + XGB Ensemble)
  Module 3 → CIC-MalMem-2022 (CNN, 4 classes!)
  Module 4 → Davidson + DynaHate (BERT)
  Anomaly  → CICIoT2023 benign (IF + Autoencoder)


In [9]:
from xgboost import XGBClassifier
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("=" * 60)
print("   MODULE 1 — NETWORK INTRUSION DETECTION")
print("   Models: XGBoost + LSTM | Dataset: CICIoT2023")
print("=" * 60)

# Group 34 fine-grained labels into 8 dashboard categories
def simplify_iot_label(l):
    l = str(l)
    if 'Benign' in l:                          return 'Benign'
    if l.startswith('DDoS'):                   return 'DDoS'
    if l.startswith('DoS'):                    return 'DoS'
    if l.startswith('Mirai'):                  return 'Mirai_Botnet'
    if l.startswith('Recon') or 'VulnerabilityScan' in l: return 'Recon'
    if 'Spoofing' in l or 'MITM' in l:         return 'Spoofing'
    if 'BruteForce' in l:                      return 'BruteForce'
    return 'WebAttack'   # XSS, SqlInjection, CommandInjection,
                         # Backdoor_Malware, BrowserHijacking, Uploading_Attack

# Capped per-class chunked loading — without caps, DDoS floods
# (70%+ of the file) would drown out every other class
def load_capped(path, cap_per_class, chunksize=250_000):
    parts, counts = {}, {}
    for chunk in pd.read_csv(path, chunksize=chunksize):
        chunk['Attack_Type'] = chunk['label'].map(simplify_iot_label)
        for cls, grp in chunk.groupby('Attack_Type'):
            got = counts.get(cls, 0)
            if got < cap_per_class:
                take = grp.head(cap_per_class - got)
                parts.setdefault(cls, []).append(take)
                counts[cls] = got + len(take)
    df = pd.concat(
        [pd.concat(v, ignore_index=True) for v in parts.values()],
        ignore_index=True
    )
    return df

print("\nLoading train (capped 80K/class, chunked — takes a few min)...")
df_train = load_capped(NET_TRAIN_PATH, 80_000)
print(f"Train: {df_train.shape}")
print(df_train['Attack_Type'].value_counts())

print("\nLoading validation (capped 12K/class)...")
df_val = load_capped(NET_VAL_PATH, 12_000)
print("\nLoading test (capped 12K/class)...")
df_test = load_capped(NET_TEST_PATH, 12_000)
print(f"Val: {df_val.shape} | Test: {df_test.shape}")
gc.collect()

FEATURE_COLS_NET = [c for c in df_train.columns
                    if c not in ('label', 'Attack_Type')]
print(f"\nFeatures: {len(FEATURE_COLS_NET)}")

def prep(df):
    X = df[FEATURE_COLS_NET].apply(pd.to_numeric, errors='coerce')
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    return np.clip(X.values, -1e9, 1e9).astype('float32')

le_network = LabelEncoder()
y_train = le_network.fit_transform(df_train['Attack_Type'])
y_val   = le_network.transform(df_val['Attack_Type'])
y_test  = le_network.transform(df_test['Attack_Type'])
print(f"Classes: {list(le_network.classes_)}")

X_train = prep(df_train)
X_val   = prep(df_val)
X_test  = prep(df_test)
del df_train, df_val, df_test
gc.collect()

scaler_network = StandardScaler()
X_train_scaled = scaler_network.fit_transform(X_train).astype('float32')
X_val_scaled   = scaler_network.transform(X_val).astype('float32')
X_test_scaled  = scaler_network.transform(X_test).astype('float32')
del X_train, X_val, X_test
gc.collect()

print(f"\nTrain: {X_train_scaled.shape[0]:,}")
print(f"Val:   {X_val_scaled.shape[0]:,}")
print(f"Test:  {X_test_scaled.shape[0]:,}")

# ============================================
# PART 1 — XGBoost (with class weights)
# ============================================
print("\n" + "-" * 40)
print("PART 1 — Training XGBoost")
print("-" * 40)
start = time.time()

# ⚖️ PATCH: per-sample weights so rare classes (BruteForce,
# WebAttack) count more — recall on attacks matters more than
# overall accuracy for a security tool
cw = {c: len(y_train)/cnt for c, cnt in zip(*np.unique(y_train, return_counts=True))}
sw_train = np.array([cw[c] for c in y_train])
print("Class weights:",
      {le_network.classes_[c]: round(w, 1) for c, w in cw.items()})

xgb_network = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    gamma=0.1,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss',
    tree_method='hist',
    early_stopping_rounds=30
)
xgb_network.fit(
    X_train_scaled, y_train,
    sample_weight=sw_train,          # ⚖️ PATCH
    eval_set=[(X_val_scaled, y_val)],
    verbose=False
)
print(f"✅ XGBoost done in {round(time.time()-start, 2)}s")

y_pred_xgb   = xgb_network.predict(X_test_scaled)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
f1_xgb       = f1_score(y_test, y_pred_xgb, average='weighted')
print(f"XGBoost Test Accuracy: {accuracy_xgb * 100:.2f}%")
print(f"XGBoost Test F1:       {f1_xgb * 100:.2f}%")
print(classification_report(
    y_test, y_pred_xgb,
    labels=range(len(le_network.classes_)),
    target_names=le_network.classes_,
    zero_division=0
))

with open('/kaggle/working/xgb_network_model.pkl', 'wb') as f:
    pickle.dump(xgb_network, f)
xgb_network.save_model('/kaggle/working/xgb_network_model.json')
with open('/kaggle/working/le_network.pkl', 'wb') as f:
    pickle.dump(le_network, f)
with open('/kaggle/working/scaler_network.pkl', 'wb') as f:
    pickle.dump(scaler_network, f)
with open('/kaggle/working/network_feature_columns.json', 'w') as f:
    json.dump(FEATURE_COLS_NET, f, indent=2)
print("✅ XGBoost saved!")
del y_pred_xgb
gc.collect()

# ============================================
# PART 2 — LSTM (windows of 10 flows, per class)
# ============================================
print("\n" + "-" * 40)
print("PART 2 — Training LSTM")
print("-" * 40)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

SEQ_LEN = 10

# Build windows within each class (rows stay in capture order
# inside a class, so a window = a burst of similar traffic).
# Live capture equivalent: rolling buffer of last 10 flows.
def make_windows(X_scaled, y):
    Xs, ys = [], []
    for cls in np.unique(y):
        rows = X_scaled[y == cls]
        n = len(rows) // SEQ_LEN
        if n == 0:
            continue
        Xs.append(rows[:n*SEQ_LEN].reshape(n, SEQ_LEN, X_scaled.shape[1]))
        ys.append(np.full(n, cls))
    return np.concatenate(Xs), np.concatenate(ys)

Xs_train, ys_train = make_windows(X_train_scaled, y_train)
Xs_val,   ys_val   = make_windows(X_val_scaled,   y_val)
Xs_test,  ys_test  = make_windows(X_test_scaled,  y_test)
print(f"Windows — Train: {len(Xs_train):,} | Val: {len(Xs_val):,} | Test: {len(Xs_test):,}")

train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(Xs_train), torch.LongTensor(ys_train)),
    batch_size=512, shuffle=True, num_workers=2)
val_loader = DataLoader(
    TensorDataset(torch.FloatTensor(Xs_val), torch.LongTensor(ys_val)),
    batch_size=512, shuffle=False, num_workers=2)
test_loader = DataLoader(
    TensorDataset(torch.FloatTensor(Xs_test), torch.LongTensor(ys_test)),
    batch_size=512, shuffle=False, num_workers=2)
del Xs_train, Xs_val, Xs_test
gc.collect()

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size,
                            num_layers=num_layers,
                            batch_first=True, dropout=0.3)
        self.fc1     = nn.Linear(hidden_size, 64)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2     = nn.Linear(64, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out    = out[:, -1, :]
        return self.fc2(self.dropout(self.relu(self.fc1(out))))

num_classes = len(le_network.classes_)
lstm_model = LSTMClassifier(
    input_size=len(FEATURE_COLS_NET),
    hidden_size=128, num_layers=2,
    num_classes=num_classes
).to(device)
print(f"LSTM Parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            out = model(Xb.to(device))
            preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            labels.extend(yb.numpy())
    return np.array(labels), np.array(preds)

epochs = 5
best_val_acc = 0
print(f"\nTraining LSTM for {epochs} epochs...")
for epoch in range(epochs):
    lstm_model.train()
    total_loss = 0
    start = time.time()
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(lstm_model(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    v_labels, v_preds = evaluate(lstm_model, val_loader)
    val_acc = accuracy_score(v_labels, v_preds)
    print(f"  Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | "
          f"Val Acc: {val_acc*100:.2f}% | Time: {round(time.time()-start,1)}s")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(lstm_model.state_dict(),
                   '/kaggle/working/lstm_network_model.pt')

lstm_model.load_state_dict(torch.load('/kaggle/working/lstm_network_model.pt'))
t_labels, t_preds = evaluate(lstm_model, test_loader)
accuracy_lstm = accuracy_score(t_labels, t_preds)
print(f"\n✅ LSTM Test Accuracy: {accuracy_lstm * 100:.2f}%")
print(classification_report(
    t_labels, t_preds,
    labels=range(num_classes),
    target_names=le_network.classes_,
    zero_division=0
))

del X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test
gc.collect()

print(f"\n{'='*60}")
print(f"   MODULE 1!")
print(f"   XGBoost: {accuracy_xgb*100:.2f}% | LSTM: {accuracy_lstm*100:.2f}%")
print(f"{'='*60}")

   MODULE 1 — NETWORK INTRUSION DETECTION
   Models: XGBoost + LSTM | Dataset: CICIoT2023

Loading train (capped 80K/class, chunked — takes a few min)...
Train: (423509, 48)
Attack_Type
Benign          80000
DDoS            80000
DoS             80000
Mirai_Botnet    80000
Spoofing        57530
Recon           41617
WebAttack        2821
BruteForce       1541
Name: count, dtype: int64

Loading validation (capped 12K/class)...

Loading test (capped 12K/class)...
Val: (69962, 48) | Test: (69757, 48)

Features: 46
Classes: ['Benign', 'BruteForce', 'DDoS', 'DoS', 'Mirai_Botnet', 'Recon', 'Spoofing', 'WebAttack']

Train: 423,509
Val:   69,962
Test:  69,757

----------------------------------------
PART 1 — Training XGBoost
----------------------------------------
Class weights: {'Benign': np.float64(5.3), 'BruteForce': np.float64(274.8), 'DDoS': np.float64(5.3), 'DoS': np.float64(5.3), 'Mirai_Botnet': np.float64(5.3), 'Recon': np.float64(10.2), 'Spoofing': np.float64(7.4), 'WebAttack': np.f

In [22]:
import os, shutil

module2_files = [
    'rf_phishing_model.pkl',
    'xgb_phishing_model.pkl',
    'xgb_phishing_model.json',
    'le_phishing.pkl',
    'scaler_phishing.pkl',
    'phishing_feature_columns.json',
    'phishing_ensemble_config.json',
]

for f in module2_files:
    path = os.path.join('/kaggle/working', f)
    if os.path.exists(path):
        os.remove(path)
        print(f"🗑️ removed: {f}")

junk = '/kaggle/working/.virtual_documents'
if os.path.exists(junk):
    shutil.rmtree(junk)
    print("🗑️ removed: .virtual_documents/")

print("\n✅ Module 2 leftovers cleared. Remaining files:")
for f in sorted(os.listdir('/kaggle/working')):
    print(f"  {f}")

🗑️ removed: rf_phishing_model.pkl
🗑️ removed: xgb_phishing_model.pkl
🗑️ removed: xgb_phishing_model.json
🗑️ removed: le_phishing.pkl
🗑️ removed: scaler_phishing.pkl
🗑️ removed: phishing_feature_columns.json
🗑️ removed: phishing_ensemble_config.json
🗑️ removed: .virtual_documents/

✅ Module 2 leftovers cleared. Remaining files:
  ae_threshold.pkl
  autoencoder_model.pt
  bert_best_model.pt
  bert_tokenizer
  cnn_malware_model.pt
  isolation_forest.pkl
  le_malware.pkl
  le_network.pkl
  le_social.pkl
  lstm_network_model.pt
  malware_feature_columns.json
  network_feature_columns.json
  pca_anomaly.pkl
  scaler_anomaly.pkl
  scaler_malware.pkl
  scaler_network.pkl
  xgb_network_model.json
  xgb_network_model.pkl


In [23]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import math
import os

print("=" * 60)
print("   MODULE 2 — PHISHING URL DETECTION (v3.2)")
print("   Combined data + scheme-normalized features")
print("=" * 60)
gc.collect()

# ---- Auto-locate both datasets ----
phiusiil_hits, old_hits = [], []
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        p = os.path.join(dirname, filename)
        if 'phiusiil' in p.lower() and p.endswith('.csv'):
            phiusiil_hits.append(p)
        if filename.lower() == 'phishing_site_urls.csv':
            old_hits.append(p)
assert phiusiil_hits, "❌ PhiUSIIL not attached"
assert old_hits,      "❌ phishing_site_urls.csv not attached"
PHISHING_PATH, OLD_PHISHING_PATH = phiusiil_hits[0], old_hits[0]
print(f"PhiUSIIL: {PHISHING_PATH}")
print(f"Old:      {OLD_PHISHING_PATH}")

df_new = pd.read_csv(PHISHING_PATH, usecols=['URL', 'label'])
df_new['Label'] = df_new['label'].map({1: 'good', 0: 'bad'})
df_new = df_new[['URL', 'Label']]
df_old = pd.read_csv(OLD_PHISHING_PATH)
df_old['Label'] = df_old['Label'].str.lower()

df_phishing = pd.concat([df_new, df_old], ignore_index=True)
df_phishing = df_phishing.drop_duplicates(subset='URL').dropna()
df_phishing = df_phishing.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Combined: {df_phishing.shape}")
print(df_phishing['Label'].value_counts())
del df_new, df_old
gc.collect()

# ----- v4 extractor: scheme-normalized -----
# FIX: PhiUSIIL legit = "https://x.com" (no path), old legit =
# "x.com/path" (no scheme). The model learned "scheme + path =
# phishing". Stripping the scheme kills that artifact; https/http
# features are removed (they encode dataset source, not safety).
def extract_url_features_v4(url):
    url = str(url).strip()
    url = re.sub(r'^https?://', '', url, flags=re.IGNORECASE)  # normalize
    features = {}

    # Basic features
    features['url_length'] = len(url)
    features['num_dots'] = url.count('.')
    features['num_hyphens'] = url.count('-')
    features['num_underscores'] = url.count('_')
    features['num_slashes'] = url.count('/')
    features['num_question_marks'] = url.count('?')
    features['num_equal_signs'] = url.count('=')
    features['num_at_signs'] = url.count('@')
    features['num_ampersands'] = url.count('&')
    features['num_hash'] = url.count('#')
    features['num_percent'] = url.count('%')
    features['num_digits'] = sum(c.isdigit() for c in url)
    features['num_letters'] = sum(c.isalpha() for c in url)
    features['num_special'] = sum(not c.isalnum() for c in url)

    # Ratio features
    l = len(url) if len(url) > 0 else 1
    features['digit_ratio'] = features['num_digits'] / l
    features['letter_ratio'] = features['num_letters'] / l
    features['special_ratio'] = features['num_special'] / l
    features['vowel_ratio'] = sum(
        c in 'aeiou' for c in url.lower()) / l
    features['digit_letter_ratio'] = features['num_digits'] / (
        features['num_letters'] + 1)

    # Security features (has_https / has_http REMOVED — artifacts)
    features['has_ip'] = 1 if re.search(
        r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', url) else 0
    features['has_port'] = 1 if re.search(r':\d+', url) else 0
    features['has_double_slash'] = url.count('//')
    features['has_triple_www'] = 1 if url.count('www') > 1 else 0
    features['has_shortener'] = 1 if any(
        s in url for s in [
            'bit.ly', 'tinyurl', 'goo.gl', 't.co',
            'ow.ly', 'is.gd', 'buff.ly', 'tiny.cc'
        ]) else 0

    # Suspicious keywords count
    suspicious_words = [
        'login', 'signin', 'verify', 'secure', 'account',
        'update', 'banking', 'confirm', 'password', 'paypal',
        'ebay', 'amazon', 'apple', 'microsoft', 'google',
        'bank', 'free', 'winner', 'prize', 'click', 'urgent',
        'suspended', 'limited', 'validate', 'credential',
        'recover', 'unusual', 'suspend', 'restricted',
        'billing', 'payment', 'invoice', 'security'
    ]
    features['num_suspicious_words'] = sum(
        1 for w in suspicious_words if w in url.lower())
    features['has_suspicious_word'] = 1 if features['num_suspicious_words'] > 0 else 0

    # Brand names
    brands = [
        'paypal', 'ebay', 'amazon', 'apple', 'microsoft',
        'google', 'facebook', 'netflix', 'instagram',
        'twitter', 'linkedin', 'dropbox', 'adobe',
        'yahoo', 'outlook', 'office365'
    ]
    features['num_brand_names'] = sum(
        1 for b in brands if b in url.lower())
    features['has_brand_name'] = 1 if features['num_brand_names'] > 0 else 0

    # Domain features (no scheme now → domain is first segment)
    try:
        domain = url.split('/')[0]
        domain = domain.split(':')[0]
        features['domain_length'] = len(domain)
        features['num_subdomains'] = max(0, domain.count('.') - 1)
        features['domain_has_digit'] = 1 if any(
            c.isdigit() for c in domain) else 0
        features['domain_has_hyphen'] = 1 if '-' in domain else 0
        features['domain_num_hyphens'] = domain.count('-')

        tld = domain.split('.')[-1] if '.' in domain else ''
        features['tld_length'] = len(tld)
        suspicious_tlds = [
            'tk', 'ml', 'ga', 'cf', 'gq', 'xyz', 'top',
            'click', 'download', 'stream', 'gdn', 'loan',
            'men', 'work', 'party', 'date', 'racing',
            'review', 'accountant', 'science', 'faith'
        ]
        features['has_suspicious_tld'] = 1 if tld in suspicious_tlds else 0
        trusted_tlds = ['com', 'org', 'edu', 'gov', 'net', 'in', 'co']
        features['has_trusted_tld'] = 1 if tld in trusted_tlds else 0
    except:
        features['domain_length'] = 0
        features['num_subdomains'] = 0
        features['domain_has_digit'] = 0
        features['domain_has_hyphen'] = 0
        features['domain_num_hyphens'] = 0
        features['tld_length'] = 0
        features['has_suspicious_tld'] = 0
        features['has_trusted_tld'] = 0

    # Path features (no scheme → path starts after first '/')
    try:
        path = '/'.join(url.split('/')[1:])
        features['path_length'] = len(path)
        features['path_depth'] = path.count('/')
        features['path_has_exe'] = 1 if any(
            ext in path.lower() for ext in
            ['.exe', '.php', '.asp', '.jsp']) else 0
    except:
        features['path_length'] = 0
        features['path_depth'] = 0
        features['path_has_exe'] = 0

    # Query features
    features['has_query'] = 1 if '?' in url else 0
    features['query_length'] = len(url.split('?')[1]) if '?' in url else 0
    features['num_query_params'] = url.count('=')

    # Encoding features
    features['has_encoding'] = 1 if '%' in url else 0
    features['num_encoded_chars'] = url.count('%')

    # Consecutive numbers in domain
    try:
        features['has_consecutive_numbers'] = 1 if re.search(
            r'\d{4,}', url.split('/')[0]) else 0
    except:
        features['has_consecutive_numbers'] = 0

    # URL entropy
    char_freq = {}
    for c in url:
        char_freq[c] = char_freq.get(c, 0) + 1
    entropy = 0
    for freq in char_freq.values():
        p = freq / len(url) if len(url) > 0 else 1
        if p > 0:
            entropy -= p * math.log2(p)
    features['url_entropy'] = round(entropy, 4)

    return features

# the rest of the pipeline calls this name — keep both pointing to v4
extract_url_features_v3 = extract_url_features_v4
# ----- end extractor -----

print("\nExtracting features (~10 min)...")
features_list = []
urls   = df_phishing['URL'].values
labels = df_phishing['Label'].values
for i, url in enumerate(urls):
    features_list.append(extract_url_features_v4(url))
    if i % 100000 == 0:
        print(f"  {i:,} / {len(urls):,}")

df_features = pd.DataFrame(features_list).astype('float32')
PHISHING_FEATURE_COLS = list(df_features.columns)
with open('/kaggle/working/phishing_feature_columns.json', 'w') as f:
    json.dump(PHISHING_FEATURE_COLS, f, indent=2)
print(f"Total features: {df_features.shape[1]}")   # should be 46 now

le_phishing = LabelEncoder()
y_phishing  = le_phishing.fit_transform(labels)

X_temp, X_test, y_temp, y_test = train_test_split(
    df_features.values, y_phishing,
    test_size=0.15, random_state=42, stratify=y_phishing)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.1765, random_state=42, stratify=y_temp)
del df_phishing, df_features, X_temp, y_temp, features_list
gc.collect()

scaler_phishing = StandardScaler()
X_train_scaled  = scaler_phishing.fit_transform(X_train)
X_val_scaled    = scaler_phishing.transform(X_val)
X_test_scaled   = scaler_phishing.transform(X_test)
del X_train, X_val, X_test
gc.collect()
print(f"Train: {X_train_scaled.shape[0]:,} | Val: {X_val_scaled.shape[0]:,} | Test: {X_test_scaled.shape[0]:,}")

print("\nTraining Random Forest...")
start = time.time()
rf_phishing = RandomForestClassifier(
    n_estimators=300, max_depth=25,
    min_samples_split=3, min_samples_leaf=1,
    random_state=42, n_jobs=-1, class_weight='balanced')
rf_phishing.fit(X_train_scaled, y_train)
print(f"✅ RF done in {round(time.time()-start, 2)}s")

spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Computed scale_pos_weight: {spw:.3f}")
print("Training XGBoost...")
start = time.time()
xgb_phishing = XGBClassifier(
    n_estimators=500, max_depth=10, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85,
    min_child_weight=1, gamma=0.05,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1,
    eval_metric='logloss', tree_method='hist',
    scale_pos_weight=spw, early_stopping_rounds=30)
xgb_phishing.fit(X_train_scaled, y_train,
                 eval_set=[(X_val_scaled, y_val)], verbose=False)
print(f"✅ XGBoost done in {round(time.time()-start, 2)}s")

rf_val_proba  = rf_phishing.predict_proba(X_val_scaled)
xgb_val_proba = xgb_phishing.predict_proba(X_val_scaled)
best_w, best_val_acc = 0.5, 0
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    acc = accuracy_score(y_val, np.argmax(
        rf_val_proba*(1-w) + xgb_val_proba*w, axis=1))
    print(f"  XGB weight {w} → Val Acc: {acc*100:.2f}%")
    if acc > best_val_acc:
        best_val_acc, best_w = acc, w
print(f"Best XGB weight: {best_w}")

ensemble_pred = np.argmax(
    rf_phishing.predict_proba(X_test_scaled)*(1-best_w) +
    xgb_phishing.predict_proba(X_test_scaled)*best_w, axis=1)
accuracy_ph = accuracy_score(y_test, ensemble_pred)
f1_ph       = f1_score(y_test, ensemble_pred, average='weighted')
print(f"\nEnsemble Test Accuracy: {accuracy_ph*100:.2f}%")
print(f"Ensemble Test F1:       {f1_ph*100:.2f}%")
print(classification_report(y_test, ensemble_pred,
                            target_names=le_phishing.classes_))

with open('/kaggle/working/rf_phishing_model.pkl', 'wb') as f:
    pickle.dump(rf_phishing, f)
with open('/kaggle/working/xgb_phishing_model.pkl', 'wb') as f:
    pickle.dump(xgb_phishing, f)
xgb_phishing.save_model('/kaggle/working/xgb_phishing_model.json')
with open('/kaggle/working/le_phishing.pkl', 'wb') as f:
    pickle.dump(le_phishing, f)
with open('/kaggle/working/scaler_phishing.pkl', 'wb') as f:
    pickle.dump(scaler_phishing, f)
with open('/kaggle/working/phishing_ensemble_config.json', 'w') as f:
    json.dump({'xgb_weight': best_w}, f)

del X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test
gc.collect()
print(f"\n✅ Module 2 (v3.2 normalized) Complete! Accuracy: {accuracy_ph*100:.2f}%")

   MODULE 2 — PHISHING URL DETECTION (v3.2)
   Combined data + scheme-normalized features
PhiUSIIL: /kaggle/input/datasets/ndarvind/phiusiil-phishing-url-dataset/PhiUSIIL_Phishing_URL_Dataset.csv
Old:      /kaggle/input/datasets/taruntiwarihp/phishing-site-urls/phishing_site_urls.csv
Combined: (742565, 2)
Label
good    527747
bad     214818
Name: count, dtype: int64

Extracting features (~10 min)...
  0 / 742,565
  100,000 / 742,565
  200,000 / 742,565
  300,000 / 742,565
  400,000 / 742,565
  500,000 / 742,565
  600,000 / 742,565
  700,000 / 742,565
Total features: 46
Train: 519,776 | Val: 111,404 | Test: 111,385

Training Random Forest...
✅ RF done in 167.15s
Computed scale_pos_weight: 0.407
Training XGBoost...
✅ XGBoost done in 33.4s
  XGB weight 0.3 → Val Acc: 90.86%
  XGB weight 0.4 → Val Acc: 90.83%
  XGB weight 0.5 → Val Acc: 90.75%
  XGB weight 0.6 → Val Acc: 90.67%
  XGB weight 0.7 → Val Acc: 90.56%
Best XGB weight: 0.3

Ensemble Test Accuracy: 90.91%
Ensemble Test F1:       9

In [13]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("=" * 60)
print("   MODULE 3 — MALWARE CLASSIFICATION (v3)")
print("   CNN | Benign / Ransomware / Spyware / Trojan")
print("=" * 60)
gc.collect()

df_mal = pd.read_csv(MALWARE_PATH)
print(f"Shape: {df_mal.shape}")

# Category looks like 'Ransomware-Shade-...' → family = first part
df_mal['Family'] = df_mal['Category'].astype(str).str.split('-').str[0]
print(df_mal['Family'].value_counts())

MALWARE_FEATURE_COLS = [c for c in df_mal.columns
                        if c not in ('Category', 'Class', 'Family')]
with open('/kaggle/working/malware_feature_columns.json', 'w') as f:
    json.dump(MALWARE_FEATURE_COLS, f, indent=2)
print(f"Features: {len(MALWARE_FEATURE_COLS)}")

le_malware = LabelEncoder()
y_malware  = le_malware.fit_transform(df_mal['Family'])
print(f"Classes: {list(le_malware.classes_)}")

X_malware = df_mal[MALWARE_FEATURE_COLS].apply(
    pd.to_numeric, errors='coerce').fillna(0).astype('float32').values

X_temp, X_test, y_temp, y_test = train_test_split(
    X_malware, y_malware,
    test_size=0.15, random_state=42, stratify=y_malware)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.1765, random_state=42, stratify=y_temp)
del df_mal, X_malware, X_temp, y_temp
gc.collect()

scaler_malware = StandardScaler()
X_train_scaled = scaler_malware.fit_transform(X_train).astype('float32')
X_val_scaled   = scaler_malware.transform(X_val).astype('float32')
X_test_scaled  = scaler_malware.transform(X_test).astype('float32')
del X_train, X_val, X_test
gc.collect()
print(f"Train: {X_train_scaled.shape[0]:,} | Val: {X_val_scaled.shape[0]:,} | Test: {X_test_scaled.shape[0]:,}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def make_loader(X, y, shuffle):
    return DataLoader(
        TensorDataset(torch.FloatTensor(X.reshape(-1, 1, X.shape[1])),
                      torch.LongTensor(y)),
        batch_size=64, shuffle=shuffle, num_workers=2)

train_loader = make_loader(X_train_scaled, y_train, True)
val_loader   = make_loader(X_val_scaled,   y_val,   False)
test_loader  = make_loader(X_test_scaled,  y_test,  False)

n_feats = X_train_scaled.shape[1]
del X_train_scaled, X_val_scaled, X_test_scaled
gc.collect()

class MalwareCNN(nn.Module):
    def __init__(self, input_length, num_classes):
        super(MalwareCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.pool    = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.3)
        conv_out = input_length // 8
        self.fc1 = nn.Linear(128 * conv_out, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()
        self.bn1, self.bn2, self.bn3 = (nn.BatchNorm1d(32),
                                        nn.BatchNorm1d(64),
                                        nn.BatchNorm1d(128))

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        return self.fc3(x)

cnn_model = MalwareCNN(input_length=n_feats,
                       num_classes=len(le_malware.classes_)).to(device)
print(f"CNN Parameters: {sum(p.numel() for p in cnn_model.parameters()):,}")

class_counts  = np.bincount(y_train)
print(f"Train class counts: {class_counts}")
class_weights = torch.FloatTensor(
    [len(y_train)/c for c in class_counts]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=2, factor=0.5)

def evaluate_cnn(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            out = model(Xb.to(device))
            preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            labels.extend(yb.numpy())
    return np.array(labels), np.array(preds)

epochs, best_val = 30, 0      # was 10
print(f"\nTraining CNN for {epochs} epochs...")
for epoch in range(epochs):
    cnn_model.train()
    total_loss = 0
    start = time.time()
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(cnn_model(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(cnn_model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    scheduler.step(avg_loss)

    v_labels, v_preds = evaluate_cnn(cnn_model, val_loader)
    val_acc = accuracy_score(v_labels, v_preds)
    print(f"  Epoch {epoch+1:2d}/{epochs} | Loss: {avg_loss:.4f} | "
          f"Val Acc: {val_acc*100:.2f}% | Time: {round(time.time()-start,1)}s")
    if val_acc > best_val:
        best_val = val_acc
        torch.save(cnn_model.state_dict(),
                   '/kaggle/working/cnn_malware_model.pt')

cnn_model.load_state_dict(torch.load('/kaggle/working/cnn_malware_model.pt'))
t_labels, t_preds = evaluate_cnn(cnn_model, test_loader)
accuracy_mal = accuracy_score(t_labels, t_preds)
print(f"\n✅ CNN Test Accuracy: {accuracy_mal*100:.2f}%")
print(classification_report(t_labels, t_preds,
                            target_names=le_malware.classes_,
                            zero_division=0))

with open('/kaggle/working/le_malware.pkl', 'wb') as f:
    pickle.dump(le_malware, f)
with open('/kaggle/working/scaler_malware.pkl', 'wb') as f:
    pickle.dump(scaler_malware, f)
print(f"\n✅ Module 3 (v3) Complete!")
gc.collect()

   MODULE 3 — MALWARE CLASSIFICATION (v3)
   CNN | Benign / Ransomware / Spyware / Trojan
Shape: (58596, 57)
Family
Benign        29298
Spyware       10020
Ransomware     9791
Trojan         9487
Name: count, dtype: int64
Features: 55
Classes: ['Benign', 'Ransomware', 'Spyware', 'Trojan']
Train: 41,015 | Val: 8,791 | Test: 8,790
CNN Parameters: 249,220
Train class counts: [20507  6853  7014  6641]

Training CNN for 30 epochs...
  Epoch  1/30 | Loss: 0.7844 | Val Acc: 75.53% | Time: 4.6s
  Epoch  2/30 | Loss: 0.7227 | Val Acc: 76.17% | Time: 4.6s
  Epoch  3/30 | Loss: 0.6877 | Val Acc: 78.61% | Time: 4.7s
  Epoch  4/30 | Loss: 0.6627 | Val Acc: 78.90% | Time: 4.6s
  Epoch  5/30 | Loss: 0.6454 | Val Acc: 78.83% | Time: 4.7s
  Epoch  6/30 | Loss: 0.6344 | Val Acc: 79.42% | Time: 4.7s
  Epoch  7/30 | Loss: 0.6251 | Val Acc: 78.74% | Time: 4.7s
  Epoch  8/30 | Loss: 0.6188 | Val Acc: 80.10% | Time: 4.6s
  Epoch  9/30 | Loss: 0.6111 | Val Acc: 80.47% | Time: 4.5s
  Epoch 10/30 | Loss: 0.6015

160

In [12]:
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

print("=" * 60)
print("   MODULE 4 — SOCIAL MEDIA CRIME DETECTION (v3)")
print("   Model: BERT | Data: Davidson + DynaHate combined")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU")

gc.collect()

# ---------- Part A: Davidson ----------
df_social = pd.read_csv(SOCIAL_PATH)
label_map = {0: 'Hate Speech', 1: 'Offensive Language', 2: 'Normal'}
df_social['label_name'] = df_social['class'].map(label_map)
df_davidson = df_social[['tweet', 'label_name']].rename(
    columns={'tweet': 'text'})
print(f"Davidson: {df_davidson.shape[0]:,} samples")

# ---------- Part B: DynaHate (implicit hate fix) ----------
print("\nDownloading DynaHate from GitHub...")
df_dyna = pd.read_csv(DYNAHATE_URL)
df_dyna.columns = [c.lower() for c in df_dyna.columns]
df_dyna['label_name'] = df_dyna['label'].map({
    'hate':    'Hate Speech',
    'nothate': 'Normal'
})
df_dyna = df_dyna[['text', 'label_name']].dropna()
print(f"DynaHate: {df_dyna.shape[0]:,} samples")

# ---------- Combine ----------
df_combined = pd.concat([df_davidson, df_dyna], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset='text').dropna()
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"\nCombined: {df_combined.shape[0]:,} samples")
print(df_combined['label_name'].value_counts())

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'\brt\b', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_combined['clean_text'] = df_combined['text'].apply(clean_text)
df_combined = df_combined[df_combined['clean_text'].str.len() > 0]

le_social = LabelEncoder()
y_social  = le_social.fit_transform(df_combined['label_name'])
print(f"Classes: {list(le_social.classes_)}")

texts = df_combined['clean_text'].values

X_temp, X_test_bert, y_temp, y_test_bert = train_test_split(
    texts, y_social,
    test_size=0.15, random_state=42, stratify=y_social)
X_train_bert, X_val_bert, y_train_bert, y_val_bert = train_test_split(
    X_temp, y_temp,
    test_size=0.1765, random_state=42, stratify=y_temp)

print(f"Train: {len(X_train_bert):,} | Val: {len(X_val_bert):,} | Test: {len(X_test_bert):,}")
del df_social, df_davidson, df_dyna, df_combined, X_temp, y_temp
gc.collect()

print("\nLoading BERT tokenizer...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("✅ Tokenizer loaded!")

class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(
                self.labels[idx], dtype=torch.long
            )
        }

train_loader = DataLoader(
    TweetDataset(X_train_bert, y_train_bert, tokenizer),
    batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(
    TweetDataset(X_val_bert, y_val_bert, tokenizer),
    batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(
    TweetDataset(X_test_bert, y_test_bert, tokenizer),
    batch_size=32, shuffle=False, num_workers=2)

print("\nLoading BERT model...")
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(le_social.classes_)
).to(device)
print(f"✅ BERT loaded!")

epochs      = 3
optimizer   = AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps)

def evaluate_bert(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            outputs = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device))
            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            labels.extend(batch['label'].numpy())
    return np.array(labels), np.array(preds)

print(f"\nTraining BERT for {epochs} epochs (~20 min each)...")
print("=" * 60)
best_val = 0

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    print("-" * 30)
    bert_model.train()
    total_loss = 0
    start      = time.time()

    for idx, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_batch   = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        if idx % 200 == 0:
            print(f"  Batch {idx}/{len(train_loader)} "
                  f"| Loss: {loss.item():.4f} "
                  f"| Time: {round(time.time()-start,1)}s")

    avg_loss = total_loss / len(train_loader)
    v_labels, v_preds = evaluate_bert(bert_model, val_loader)
    val_acc = accuracy_score(v_labels, v_preds)
    val_f1  = f1_score(v_labels, v_preds, average='weighted')
    print(f"\n  Loss:    {avg_loss:.4f}")
    print(f"  Val Acc: {val_acc * 100:.2f}%")
    print(f"  Val F1:  {val_f1 * 100:.2f}%")

    if val_acc > best_val:
        best_val = val_acc
        torch.save(bert_model.state_dict(),
                   '/kaggle/working/bert_best_model.pt')
        print(f"  ✅ Best model saved!")

bert_model.load_state_dict(torch.load('/kaggle/working/bert_best_model.pt'))
t_labels, t_preds = evaluate_bert(bert_model, test_loader)
accuracy_soc = accuracy_score(t_labels, t_preds)
f1_soc       = f1_score(t_labels, t_preds, average='weighted')

print(f"\nTest Accuracy: {accuracy_soc * 100:.2f}%")
print(f"Test F1:       {f1_soc * 100:.2f}%")
print(classification_report(t_labels, t_preds,
                            target_names=le_social.classes_))

tokenizer.save_pretrained('/kaggle/working/bert_tokenizer')
with open('/kaggle/working/le_social.pkl', 'wb') as f:
    pickle.dump(le_social, f)

print("\n✅ Module 4 (v3) Complete!")
gc.collect()

   MODULE 4 — SOCIAL MEDIA CRIME DETECTION (v3)
   Model: BERT | Data: Davidson + DynaHate combined
GPU: Tesla T4
Davidson: 24,783 samples

DynaHate: 41,144 samples

Combined: 65,917 samples
label_name
Hate Speech           23598
Normal                23129
Offensive Language    19190
Name: count, dtype: int64
Classes: ['Hate Speech', 'Normal', 'Offensive Language']
Train: 46,138 | Val: 9,889 | Test: 9,888



Loading BERT tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ BERT loaded!

Training BERT for 3 epochs (~20 min each)...

Epoch 1/3
------------------------------
  Batch 0/1442 | Loss: 1.0672 | Time: 1.1s
  Batch 200/1442 | Loss: 0.7134 | Time: 137.8s
  Batch 400/1442 | Loss: 0.3719 | Time: 273.1s
  Batch 600/1442 | Loss: 0.4663 | Time: 408.2s
  Batch 800/1442 | Loss: 0.6094 | Time: 543.6s
  Batch 1000/1442 | Loss: 0.5646 | Time: 679.0s
  Batch 1200/1442 | Loss: 0.3454 | Time: 814.3s
  Batch 1400/1442 | Loss: 0.3718 | Time: 949.5s

  Loss:    0.5665
  Val Acc: 81.52%
  Val F1:  81.63%
  ✅ Best model saved!

Epoch 2/3
------------------------------
  Batch 0/1442 | Loss: 0.3841 | Time: 0.8s
  Batch 200/1442 | Loss: 0.4451 | Time: 136.1s
  Batch 400/1442 | Loss: 0.3904 | Time: 271.5s
  Batch 600/1442 | Loss: 0.1925 | Time: 406.5s
  Batch 800/1442 | Loss: 0.2014 | Time: 541.9s
  Batch 1000/1442 | Loss: 0.2983 | Time: 677.1s
  Batch 1200/1442 | Loss: 0.2441 | Time: 812.5s
  Batch 1400/1442 | Loss: 0.2298 | Time: 948.0s

  Loss:    0.3515
  Val Acc

98

In [14]:
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("=" * 60)
print("   ANOMALY DETECTION (v3) — IF + Autoencoder")
print("=" * 60)

print("Loading benign (train) + mixed val/test (capped per class)...")
df_b_train = load_capped(NET_TRAIN_PATH, 100_000)
df_b_train = df_b_train[df_b_train['Attack_Type'] == 'Benign']
df_v = load_capped(NET_VAL_PATH, 5_000)
df_t = load_capped(NET_TEST_PATH, 5_000)

def to_X(df):
    X = df[FEATURE_COLS_NET].apply(pd.to_numeric, errors='coerce')
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    return np.clip(X.values, -1e9, 1e9).astype('float32')

X_b_train = to_X(df_b_train)
X_b_val   = to_X(df_v[df_v['Attack_Type'] == 'Benign'])
X_a_val   = to_X(df_v[df_v['Attack_Type'] != 'Benign'])
X_b_test  = to_X(df_t[df_t['Attack_Type'] == 'Benign'])
X_a_test  = to_X(df_t[df_t['Attack_Type'] != 'Benign'])
del df_b_train, df_v, df_t
gc.collect()
print(f"Benign train: {len(X_b_train):,}")
print(f"Val   benign/attack: {len(X_b_val):,} / {len(X_a_val):,}")
print(f"Test  benign/attack: {len(X_b_test):,} / {len(X_a_test):,}")

scaler_anomaly = StandardScaler()
X_b_train_s = scaler_anomaly.fit_transform(X_b_train)
pca_anomaly = PCA(n_components=20, random_state=42)
X_b_train_pca = pca_anomaly.fit_transform(X_b_train_s)
X_b_val_pca  = pca_anomaly.transform(scaler_anomaly.transform(X_b_val))
X_a_val_pca  = pca_anomaly.transform(scaler_anomaly.transform(X_a_val))
X_b_test_pca = pca_anomaly.transform(scaler_anomaly.transform(X_b_test))
X_a_test_pca = pca_anomaly.transform(scaler_anomaly.transform(X_a_test))
del X_b_train, X_b_val, X_a_val, X_b_test, X_a_test, X_b_train_s
gc.collect()
print(f"PCA variance: {pca_anomaly.explained_variance_ratio_.sum()*100:.1f}%")

# ---- Isolation Forest: fit on train, tune on val, report on test ----
print("\n--- Isolation Forest ---")
best_score, best_if, best_cont = 0, None, 0
for contamination in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]:
    model = IsolationForest(
        n_estimators=500, contamination=contamination,
        random_state=42, n_jobs=-1,
        max_samples=min(len(X_b_train_pca), 15000))
    model.fit(X_b_train_pca)
    b_acc = (model.predict(X_b_val_pca[:2000]) == 1).mean()
    a_det = (model.predict(X_a_val_pca[:2000]) == -1).mean()
    score = b_acc*0.4 + a_det*0.6
    print(f"  contamination={contamination} → Normal: {b_acc*100:.1f}% | Attacks: {a_det*100:.1f}%")
    if score > best_score:
        best_score, best_if, best_cont = score, model, contamination
isolation_forest = best_if
print(f"Best contamination: {best_cont}")

if_normal_pct  = (isolation_forest.predict(X_b_test_pca[:2000]) == 1).mean()*100
if_attacks_pct = (isolation_forest.predict(X_a_test_pca[:2000]) == -1).mean()*100
print(f"TEST Normal: {if_normal_pct:.1f}% | TEST Attacks: {if_attacks_pct:.1f}%")

with open('/kaggle/working/isolation_forest.pkl', 'wb') as f:
    pickle.dump(isolation_forest, f)
with open('/kaggle/working/scaler_anomaly.pkl', 'wb') as f:
    pickle.dump(scaler_anomaly, f)
with open('/kaggle/working/pca_anomaly.pkl', 'wb') as f:
    pickle.dump(pca_anomaly, f)

# ---- Autoencoder ----
print("\n--- Autoencoder ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class DeepAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(DeepAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2),
            nn.Dropout(0.2),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.LeakyReLU(0.2),
            nn.Linear(32, 16), nn.BatchNorm1d(16), nn.LeakyReLU(0.2),
            nn.Linear(16, 8))
        self.decoder = nn.Sequential(
            nn.Linear(8, 16), nn.BatchNorm1d(16), nn.LeakyReLU(0.2),
            nn.Linear(16, 32), nn.BatchNorm1d(32), nn.LeakyReLU(0.2),
            nn.Dropout(0.2),
            nn.Linear(32, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2),
            nn.Linear(64, input_dim))

    def forward(self, x):
        return self.decoder(self.encoder(x))

dataloader = DataLoader(TensorDataset(torch.FloatTensor(X_b_train_pca)),
                        batch_size=512, shuffle=True, num_workers=2)
autoencoder  = DeepAutoencoder(input_dim=20).to(device)
ae_optimizer = torch.optim.Adam(autoencoder.parameters(),
                                lr=0.001, weight_decay=1e-5)
ae_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ae_optimizer, T_max=100)
ae_criterion = nn.MSELoss()

best_loss = float('inf')
print("Training for 100 epochs...")
for epoch in range(100):
    autoencoder.train()
    total_loss = 0
    for (Xb,) in dataloader:
        Xb = Xb.to(device)
        ae_optimizer.zero_grad()
        loss = ae_criterion(autoencoder(Xb), Xb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(autoencoder.parameters(), 1.0)
        ae_optimizer.step()
        total_loss += loss.item()
    ae_scheduler.step()
    avg_loss = total_loss / len(dataloader)
    if (epoch+1) % 20 == 0:
        print(f"  Epoch {epoch+1:3d}/100 | Loss: {avg_loss:.6f}")
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(autoencoder.state_dict(),
                   '/kaggle/working/autoencoder_model.pt')

autoencoder.load_state_dict(torch.load('/kaggle/working/autoencoder_model.pt'))
autoencoder.eval()

def recon_errors(X, limit=None):
    if limit: X = X[:limit]
    errors = []
    for i in range(0, len(X), 512):
        b = torch.FloatTensor(X[i:i+512]).to(device)
        with torch.no_grad():
            errors.extend(torch.mean((autoencoder(b)-b)**2, dim=1).cpu().numpy())
    return np.array(errors)

b_val_err = recon_errors(X_b_val_pca)
a_val_err = recon_errors(X_a_val_pca, limit=5000)
print(f"\nBenign VAL mean err: {b_val_err.mean():.6f} | Attack VAL mean err: {a_val_err.mean():.6f}")

best_thresh_score, best_threshold, best_pct = 0, 0, 0
for pct in [70, 75, 80, 85, 90, 95, 99]:
    th = np.percentile(b_val_err, pct)
    b_ok = (b_val_err <= th).mean()
    a_ok = (a_val_err > th).mean()
    score = b_ok*0.4 + a_ok*0.6
    print(f"  {pct}th ({th:.6f}) → Normal: {b_ok*100:.1f}% | Attacks: {a_ok*100:.1f}%")
    if score > best_thresh_score:
        best_thresh_score, best_threshold, best_pct = score, th, pct
print(f"Best threshold: {best_threshold:.6f} ({best_pct}th percentile)")

b_test_err = recon_errors(X_b_test_pca)
a_test_err = recon_errors(X_a_test_pca, limit=5000)
ae_normal_pct = (b_test_err <= best_threshold).mean()*100
ae_attack_pct = (a_test_err > best_threshold).mean()*100

with open('/kaggle/working/ae_threshold.pkl', 'wb') as f:
    pickle.dump(best_threshold, f)

if_test_preds = isolation_forest.predict(X_a_test_pca[:5000])
n_comb = min(len(if_test_preds), len(a_test_err))
combined_pct = np.mean([(if_test_preds[i] == -1) or (a_test_err[i] > best_threshold)
                        for i in range(n_comb)])*100

anomaly_combined_pct = combined_pct
print(f"\n{'='*60}")
print(f"   ANOMALY (v3) — HONEST TEST RESULTS")
print(f"{'='*60}")
print(f"IF:  Normal {if_normal_pct:.1f}% | Attacks {if_attacks_pct:.1f}%")
print(f"AE:  Normal {ae_normal_pct:.1f}% | Attacks {ae_attack_pct:.1f}%")
print(f"Combined attack detection: {combined_pct:.1f}%")

del X_b_train_pca, X_b_val_pca, X_a_val_pca, X_b_test_pca, X_a_test_pca
gc.collect()
print("\n🎉 ANOMALY DETECTION COMPLETE!")

   ANOMALY DETECTION (v3) — IF + Autoencoder
Loading benign (train) + mixed val/test (capped per class)...
Benign train: 100,000
Val   benign/attack: 5,000 / 25,947
Test  benign/attack: 5,000 / 25,945
PCA variance: 94.4%

--- Isolation Forest ---
  contamination=0.05 → Normal: 94.8% | Attacks: 76.8%
  contamination=0.1 → Normal: 89.1% | Attacks: 88.4%
  contamination=0.15 → Normal: 85.0% | Attacks: 89.8%
  contamination=0.2 → Normal: 79.8% | Attacks: 91.2%
  contamination=0.25 → Normal: 74.9% | Attacks: 92.3%
  contamination=0.3 → Normal: 69.7% | Attacks: 93.5%
Best contamination: 0.1
TEST Normal: 90.2% | TEST Attacks: 86.9%

--- Autoencoder ---
Training for 100 epochs...
  Epoch  20/100 | Loss: 0.285059
  Epoch  40/100 | Loss: 0.245752
  Epoch  60/100 | Loss: 0.203754
  Epoch  80/100 | Loss: 0.183979
  Epoch 100/100 | Loss: 0.185424

Benign VAL mean err: 0.047758 | Attack VAL mean err: 611.125977
  70th (0.038147) → Normal: 70.0% | Attacks: 98.4%
  75th (0.043396) → Normal: 75.0% | At

In [24]:
# ============================================
# V3 REAL-LIFE TEST — ALL 5 MODULES
# ============================================
import random
random.seed(7)

# ---------- helpers ----------
TRUSTED_DOMAINS = ['google.com', 'youtube.com', 'facebook.com', 'instagram.com',
    'twitter.com', 'linkedin.com', 'netflix.com', 'amazon.com', 'microsoft.com',
    'apple.com', 'github.com', 'stackoverflow.com', 'wikipedia.org', 'reddit.com',
    'sbi.co.in', 'hdfcbank.com', 'icicibank.com', 'kaggle.com']

def predict_url_ensemble(url):
    url_str = str(url)
    try:
        domain = url_str.split('/')[2] if '//' in url_str else url_str.split('/')[0]
        domain = domain.replace('www.', '').split(':')[0]
        for trusted in TRUSTED_DOMAINS:
            if domain == trusted or domain.endswith('.' + trusted):
                return 'SAFE', 99.0
    except:
        pass
    feats = extract_url_features_v3(url_str)
    arr = np.array([feats[c] for c in PHISHING_FEATURE_COLS]).reshape(1, -1).astype('float32')
    arr_s = scaler_phishing.transform(arr)
    proba = (rf_phishing.predict_proba(arr_s)*(1-best_w) +
             xgb_phishing.predict_proba(arr_s)*best_w)
    pred = np.argmax(proba, axis=1)[0]
    conf = round(float(np.max(proba))*100, 2)
    return ('SAFE' if le_phishing.classes_[pred] == 'good' else 'PHISHING'), conf

def predict_social(text):
    enc = tokenizer(clean_text(text), max_length=128, padding='max_length',
                    truncation=True, return_tensors='pt')
    bert_model.eval()
    with torch.no_grad():
        out = bert_model(input_ids=enc['input_ids'].to(device),
                         attention_mask=enc['attention_mask'].to(device))
    probs = torch.softmax(out.logits, dim=1)
    pred  = torch.argmax(probs, dim=1).item()
    return le_social.classes_[pred], round(probs[0][pred].item()*100, 2)

# ---------- 📡 MODULE 1: unseen flows, one per class ----------
print("=" * 60)
print("📡 MODULE 1 — Real network flows (one per attack type)")
print("=" * 60)
df_rl = load_capped(NET_TEST_PATH, 50)
X_rl = df_rl[FEATURE_COLS_NET].apply(pd.to_numeric, errors='coerce').fillna(0)
X_rl = np.clip(X_rl.values, -1e9, 1e9).astype('float32')
X_rl_s = scaler_network.transform(X_rl)
actual_rl = df_rl['Attack_Type'].values

correct, total = 0, 0
for cls in le_network.classes_:
    idxs = [i for i in range(len(actual_rl)) if actual_rl[i] == cls][:2]
    for idx in idxs:
        pred = le_network.classes_[xgb_network.predict(X_rl_s[idx].reshape(1, -1))[0]]
        icon = "✅" if pred == actual_rl[idx] else "❌"
        correct += (pred == actual_rl[idx]); total += 1
        print(f"{icon} Predicted: {pred:14s} | Actual: {actual_rl[idx]}")
print(f"\nNetwork score: {correct}/{total}")

# ---------- 🌐 MODULE 2: incl. HTTPS-phishing stress test ----------
print("\n" + "=" * 60)
print("🌐 MODULE 2 — URLs (incl. https phishing + non-whitelist legit)")
print("=" * 60)
test_urls = [
    ("https://www.zomato.com/bangalore/restaurants",            "SAFE"),
    ("https://www.bbc.com/news/technology",                     "SAFE"),
    ("https://pytorch.org/docs/stable/nn.html",                 "SAFE"),
    ("https://www.swiggy.com/offers",                           "SAFE"),
    ("https://secure-sbi-netbanking-update.tk/login/verify",    "PHISHING"),  # https!
    ("https://amaz0n-prize-winner2026.xyz/claim-iphone",        "PHISHING"),  # https!
    ("https://netflix-payment-failed.click/billing/update",     "PHISHING"),  # https!
    ("http://103.94.122.8/icici/secure/password-reset",         "PHISHING"),
    ("http://bit.ly.account-verify-paytm.ml/kyc-update",        "PHISHING"),
    ("https://login-verification-microsoft.support/account",    "PHISHING"),  # https!
]
correct = 0
for url, expected in test_urls:
    label, conf = predict_url_ensemble(url)
    icon = "✅" if label == expected else "❌"
    correct += (label == expected)
    print(f"{icon} {label:8s} ({conf:5.1f}%) → {url[:52]}")
print(f"\nPhishing score: {correct}/10  (https-phishing rows = HTTPS-shortcut check)")

# ---------- 🦠 MODULE 3: family detection ----------
print("\n" + "=" * 60)
print("🦠 MODULE 3 — Malware families")
print("=" * 60)
df_m = pd.read_csv(MALWARE_PATH)
df_m['Family'] = df_m['Category'].astype(str).str.split('-').str[0]
X_m = df_m[MALWARE_FEATURE_COLS].apply(pd.to_numeric, errors='coerce').fillna(0).astype('float32').values
X_m_s = scaler_malware.transform(X_m)
cnn_model.eval()
correct, total = 0, 0
for fam in le_malware.classes_:
    idxs = random.sample([i for i in range(len(df_m)) if df_m['Family'].iloc[i] == fam], 3)
    for idx in idxs:
        sample = torch.FloatTensor(X_m_s[idx].reshape(1, 1, -1)).to(device)
        with torch.no_grad():
            pred = torch.argmax(cnn_model(sample), dim=1).item()
        pred_f = le_malware.classes_[pred]
        ok_detect = (fam == 'Benign') == (pred_f == 'Benign')
        icon = "✅" if pred_f == fam else ("🟡" if ok_detect else "❌")
        correct += (pred_f == fam); total += 1
        print(f"{icon} Predicted: {pred_f:11s} | Actual: {fam}")
print(f"\nFamily score: {correct}/{total} (🟡 = malware caught, family wrong = still alerts!)")
del df_m, X_m, X_m_s
gc.collect()

# ---------- 💬 MODULE 4: THE REVENGE ROUND ----------
print("\n" + "=" * 60)
print("💬 MODULE 4 — incl. the 4 sentences the OLD model called 'Normal'")
print("=" * 60)
test_social = [
    ("bro the new spiderman movie was insane go watch it",     "Normal"),
    ("finally finished my ML project after 3 weeks",           "Normal"),
    ("congrats on the new job dude you deserve it",            "Normal"),
    ("you are such a dumb idiot i cant believe this",          "Offensive Language"),
    ("stop talking crap nobody asked your opinion",            "Offensive Language"),
    # ↓ the old model said "Normal" (91-95% confidence) on ALL of these ↓
    ("those people are animals and should all be thrown out",  "Hate Speech"),
    ("that community is ruining our country send them back",   "Hate Speech"),
    ("people of that religion are all terrorists",             "Hate Speech"),
    ("women belong in the kitchen not in offices",             "Hate Speech"),
    ("immigrants are parasites living off our taxes",          "Hate Speech"),
]
correct = 0
for text, expected in test_social:
    label, conf = predict_social(text)
    icon = "✅" if label == expected else "❌"
    correct += (label == expected)
    flag = "✅" if label == 'Normal' else ("⚠️" if label == 'Offensive Language' else "🚨")
    print(f"{icon} {flag} {label:18s} ({conf:5.1f}%) | {text[:48]}")
print(f"\nSocial score: {correct}/10")

# ---------- 🔍 ANOMALY ----------
print("\n" + "=" * 60)
print("🔍 ANOMALY — zero-day simulation")
print("=" * 60)
df_an = load_capped(NET_TEST_PATH, 30)
ben = df_an[df_an['Attack_Type'] == 'Benign'][FEATURE_COLS_NET].head(5)
att = df_an[df_an['Attack_Type'] != 'Benign'][FEATURE_COLS_NET].sample(5, random_state=7)

def anomaly_flags(df_rows):
    X = df_rows.apply(pd.to_numeric, errors='coerce').fillna(0)
    X = np.clip(X.values, -1e9, 1e9).astype('float32')
    X_pca = pca_anomaly.transform(scaler_anomaly.transform(X))
    if_p = isolation_forest.predict(X_pca)
    Xt = torch.FloatTensor(X_pca).to(device)
    autoencoder.eval()
    with torch.no_grad():
        err = torch.mean((autoencoder(Xt)-Xt)**2, dim=1).cpu().numpy()
    return [(if_p[i] == -1) or (err[i] > best_threshold) for i in range(len(X_pca))]

print("Normal traffic (should be SAFE):")
for f in anomaly_flags(ben):
    print(f"  {'❌ 🚨 FLAGGED' if f else '✅ SAFE'}")
print("Attack traffic (should be FLAGGED):")
for f in anomaly_flags(att):
    print(f"  {'✅ 🚨 FLAGGED' if f else '❌ missed'}")

del df_rl, df_an
gc.collect()
print("\n🎉 V3 REAL-LIFE TEST COMPLETE!")

📡 MODULE 1 — Real network flows (one per attack type)
✅ Predicted: Benign         | Actual: Benign
✅ Predicted: Benign         | Actual: Benign
✅ Predicted: BruteForce     | Actual: BruteForce
✅ Predicted: BruteForce     | Actual: BruteForce
✅ Predicted: DDoS           | Actual: DDoS
✅ Predicted: DDoS           | Actual: DDoS
✅ Predicted: DoS            | Actual: DoS
✅ Predicted: DoS            | Actual: DoS
✅ Predicted: Mirai_Botnet   | Actual: Mirai_Botnet
✅ Predicted: Mirai_Botnet   | Actual: Mirai_Botnet
✅ Predicted: Recon          | Actual: Recon
✅ Predicted: Recon          | Actual: Recon
✅ Predicted: Spoofing       | Actual: Spoofing
✅ Predicted: Spoofing       | Actual: Spoofing
✅ Predicted: WebAttack      | Actual: WebAttack
❌ Predicted: Spoofing       | Actual: WebAttack

Network score: 15/16

🌐 MODULE 2 — URLs (incl. https phishing + non-whitelist legit)
✅ SAFE     ( 92.3%) → https://www.zomato.com/bangalore/restaurants
✅ SAFE     ( 91.6%) → https://www.bbc.com/news/technolo

In [25]:
import sys, xgboost, sklearn, torch, transformers, numpy, pandas
print("python:", sys.version.split()[0])
print("xgboost:", xgboost.__version__)
print("scikit-learn:", sklearn.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)

python: 3.12.13
xgboost: 3.2.0
scikit-learn: 1.6.1
torch: 2.10.0+cu128
transformers: 5.0.0
numpy: 2.4.6
pandas: 2.3.3


In [26]:
import zipfile, os, json

metrics = {
    'network_xgboost_test_acc':   round(accuracy_xgb * 100, 2),
    'network_lstm_test_acc':      round(accuracy_lstm * 100, 2),
    'phishing_ensemble_test_acc': round(accuracy_ph * 100, 2),
    'malware_cnn_test_acc':       round(accuracy_mal * 100, 2),
    'social_bert_test_acc':       round(accuracy_soc * 100, 2),
    'anomaly_combined_detection': round(anomaly_combined_pct, 1),
}
with open('/kaggle/working/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Metrics being saved:")
for k, v in metrics.items():
    print(f"  {k}: {v}%")

zip_path = '/kaggle/working/CyberWatch_All_Models_V3.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk('/kaggle/working/'):
        for file in files:
            if file.endswith('.zip') or '.virtual_documents' in root:
                continue
            full_path = os.path.join(root, file)
            zipf.write(full_path, os.path.relpath(full_path, '/kaggle/working/'))
            print(f"  Added: {os.path.relpath(full_path, '/kaggle/working/')}")

print(f"\n✅ ZIP: {os.path.getsize(zip_path)/1024**2:.1f} MB")
print("\n🎉 CYBERWATCH AI V3 COMPLETE!")

Metrics being saved:
  network_xgboost_test_acc: 95.13%
  network_lstm_test_acc: 99.38%
  phishing_ensemble_test_acc: 90.91%
  malware_cnn_test_acc: 83.12%
  social_bert_test_acc: 83.65%
  anomaly_combined_detection: 94.2%
  Added: xgb_phishing_model.pkl
  Added: le_malware.pkl
  Added: lstm_network_model.pt
  Added: metrics.json
  Added: ae_threshold.pkl
  Added: xgb_network_model.pkl
  Added: cnn_malware_model.pt
  Added: le_social.pkl
  Added: rf_phishing_model.pkl
  Added: scaler_network.pkl
  Added: le_network.pkl
  Added: autoencoder_model.pt
  Added: phishing_feature_columns.json
  Added: scaler_phishing.pkl
  Added: network_feature_columns.json
  Added: bert_best_model.pt
  Added: le_phishing.pkl
  Added: scaler_malware.pkl
  Added: pca_anomaly.pkl
  Added: malware_feature_columns.json
  Added: xgb_phishing_model.json
  Added: xgb_network_model.json
  Added: phishing_ensemble_config.json
  Added: scaler_anomaly.pkl
  Added: isolation_forest.pkl
  Added: bert_tokenizer/tokenizer